# Pony Diffusion V6 XL - Uncensored Studio
Uses Pony Diffusion V6 XL, the ultimate model for unrestricted and adult content generation.

In [ ]:
!pip install -q diffusers transformers accelerate torch torchvision gradio safetensors

import torch
from diffusers import AutoPipelineForImage2Image
import gradio as gr

has_cuda = torch.cuda.is_available()
device = "cuda" if has_cuda else "cpu"
dtype = torch.float16 if has_cuda else torch.float32

print("Loading Pony Diffusion V6 XL (100% Uncensored)...")
# Pony V6 XL checkpoint from Civitai / HF community mirrors
pipe_i2i = AutoPipelineForImage2Image.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=dtype,
    use_safetensors=True
)

# Load Pony v6 checkpoint weights directly
from diffusers import StableDiffusionXLImg2ImgPipeline
import os

if not os.path.exists("ponyV6.safetensors"):
    print(
        "Downloading Pony Diffusion V6 XL weights..."
    )
    !wget -q -O ponyV6.safetensors "https://huggingface.co/Lunakeps/ponyDiffusionV6XL_v6StartWithThisOne/resolve/main/ponyDiffusionV6XL_v6StartWithThisOne.safetensors"

pipe_i2i = StableDiffusionXLImg2ImgPipeline.from_single_file(
    "ponyV6.safetensors",
    torch_dtype=dtype,
    use_safetensors=True
).to(device)

if hasattr(pipe_i2i, "safety_checker"):
    pipe_i2i.safety_checker = None

def edit(image, prompt, negative_prompt, strength, steps, cfg):
    if image is None:
        return None
    
    image = image.convert("RGB")
    image.thumbnail((1024, 1024))
    
    # Pony v6 requires score tags for best results
    full_prompt = "score_9, score_8_up, score_7_up, " + prompt
    full_neg = "score_6, score_5, score_4, " + negative_prompt
    
    img = pipe_i2i(
        prompt=full_prompt,
        negative_prompt=full_neg,
        image=image,
        strength=float(strength),
        num_inference_steps=int(steps),
        guidance_scale=float(cfg)
    ).images[0]
    return img

demo = gr.Interface(
    fn=edit,
    inputs=[
        gr.Image(type="pil", label="Source Image"),
        gr.Textbox(label="Prompt (Use tag format e.g. solo, expressive, ...)", value="solo, masterpiece, highly detailed"),
        gr.Textbox(label="Negative Prompt", value="source_low, lowres, bad anatomy"),
        gr.Slider(0.1, 0.6, value=0.25, label="Strength (0.2 - 0.3 preserves identity)"),
        gr.Slider(20, 50, value=30, step=1, label="Steps"),
        gr.Slider(4.0, 10.0, value=7.0, step=0.5, label="CFG Scale")
    ],
    outputs=gr.Image(label="Result (100% Uncensored)"),
    title="Pony Diffusion V6 XL Uncensored Studio"
)
demo.launch(share=True, debug=True)